# LASSO Regression

Let's continue with penalized ERM, but now use a different penalty. 

## LASSO Regression

A second important example of penalized ERM is **LASSO regression**. LASSO stands for **least absolute shrinkage and selection operator**.

Ridge regression used the squared $\ell_2$ penalty:

$$
\Omega(w) = \|w\|_2^2.
$$

LASSO instead uses the $\ell_1$ penalty:

$$
\Omega(w) = \|w\|_1.
$$

For $w \in \mathbb{R}^D$, the $\ell_1$ norm is

$$
\|w\|_1 = \sum_{j=1}^D |w_j|.
$$

We can visualize these:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

theta = np.linspace(0, 2*np.pi, 500)

# L2 ball: w_1^2 + w_2^2 <= 1
l2_x = np.cos(theta)
l2_y = np.sin(theta)

# L1 ball: |w_1| + |w_2| <= 1
l1_x = np.array([1, 0, -1, 0, 1])
l1_y = np.array([0, 1, 0, -1, 0])

fig, axes = plt.subplots(1, 2, figsize=(8, 4))

# L2 ball
axes[0].plot(l2_x, l2_y)
axes[0].fill(l2_x, l2_y, alpha=0.15)
axes[0].axhline(0, linewidth=0.8)
axes[0].axvline(0, linewidth=0.8)
axes[0].set_title(r"$\ell_2$ ball: $\|w\|_2 \leq 1$")
axes[0].set_xlabel(r"$w_1$")
axes[0].set_ylabel(r"$w_2$")
axes[0].set_aspect("equal")
axes[0].set_xlim(-1.2, 1.2)
axes[0].set_ylim(-1.2, 1.2)

# L1 ball
axes[1].plot(l1_x, l1_y)
axes[1].fill(l1_x, l1_y, alpha=0.15)
axes[1].axhline(0, linewidth=0.8)
axes[1].axvline(0, linewidth=0.8)
axes[1].set_title(r"$\ell_1$ ball: $\|w\|_1 \leq 1$")
axes[1].set_xlabel(r"$w_1$")
axes[1].set_ylabel(r"$w_2$")
axes[1].set_aspect("equal")
axes[1].set_xlim(-1.2, 1.2)
axes[1].set_ylim(-1.2, 1.2)

plt.tight_layout()
plt.show()

This gives the LASSO objective:

$$
\hat{w}^{\lambda}_{lasso}=
\arg\min_w
\frac{1}{N}\|y - Xw\|_2^2
+
\lambda \|w\|_1.
$$

As usual:

- $\lambda = 0$ gives OLS
- larger $\lambda$ shrinks coefficients more strongly
- as $\lambda \to \infty$, the solution is driven toward $0$

The key difference is that LASSO can set coefficients **exactly equal to zero**. This means LASSO does two things at once:

1. It shrinks coefficients toward zero.  
2. It performs variable selection by removing some predictors from the model.

## What changes relative to ridge?

Ridge regression solves

$$
\min_w
\frac{1}{N}\|y - Xw\|_2^2
+
\lambda \|w\|_2^2.
$$

This is smooth, and we can solve it by taking a gradient and setting it equal to zero:

$$
\hat{w}^{\lambda}_{ridge}=
(X^\top X + \lambda I)^{-1}X^\top y.
$$

LASSO solves

$$
\min_w
\frac{1}{N}\|y - Xw\|_2^2
+
\lambda \|w\|_1.
$$

The issue is that the absolute value function is not differentiable at zero. Since

$$
|w_j|
$$

has a sharp corner at $w_j=0$, the objective is convex but not smooth. So, in general, there is no simple closed-form matrix formula like ridge.

This is not just a technical nuisance. The sharp corner is exactly what allows LASSO to set coefficients equal to zero.

## Solving LASSO in practice

Since the LASSO objective is convex, but not differentiable everywhere. We cannot solve the problem by ordinary differentiation in the same way we solved ridge. 

### The subgradient idea

To handle the non-differentiability, we use the idea of a **subgradient**.

For $w_j \neq 0$, the derivative of $|w_j|$ is ordinary:

$$
\frac{d}{dw_j}|w_j|=
\begin{cases}
1, & w_j > 0, \\
-1, & w_j < 0.
\end{cases}
$$

At $w_j = 0$, the derivative does not exist. The left derivative is $-1$, while the right derivative is $1$. The **subgradient** fills in this missing idea here. 

A **subgradient** is a generalization of the derivative for convex functions that may have corners.

For a differentiable convex function, the derivative gives the slope of the tangent line. At a minimum, the derivative must be zero.

For a non-differentiable convex function, there may be many valid supporting slopes at a point. These slopes are called **subgradients**. The set of all subgradients is called the **subdifferential**.

For example, consider

$$
h(w_j) = |w_j|.
$$

Away from zero, the function is differentiable:

$$
\partial |w_j|=
\begin{cases}
\{1\}, & w_j > 0, \\
\{-1\}, & w_j < 0.
\end{cases}
$$

At zero, there is no unique tangent slope. Instead, every slope between $-1$ and $1$ defines a valid supporting line to the absolute value function. Therefore,

$$
\partial |w_j|=
[-1,1],
\qquad \text{when } w_j = 0.
$$

This is the key difference between ordinary derivatives and subgradients. The derivative is a single number. The subgradient can be a set of numbers.

The optimality condition also changes slightly. For a differentiable convex function $J(w)$, a point $\hat{w}$ minimizes $J$ if

$$
\nabla J(\hat{w}) = 0.
$$

For a convex but non-differentiable function, the analogous condition is

$$
0 \in \partial J(\hat{w}).
$$

That is, zero must belong to the set of possible subgradients at $\hat{w}$.

### Two common algorithms

There are many algorithms for solving LASSO. Two especially useful ones are:

1. **Coordinate descent**
2. **Proximal gradient descent**

Both rely on the same basic operation: **soft-thresholding**.

The soft-thresholding function is

$$
S(z,\lambda)=
\begin{cases}
z - \lambda, & z > \lambda, \\
0, & |z| \leq \lambda, \\
z + \lambda, & z < -\lambda.
\end{cases}
$$

Equivalently,

$$
S(z,\lambda)=
\text{sign}(z)(|z|-\lambda)_+.
$$

where $(\cdot)_+ = \max(\cdot,0)$ is the *positive* part. Soft-thresholding does two things at once:

- it shrinks values toward zero
- it sets small values exactly equal to zero


**Coordinate descent for LASSO**

Coordinate descent is one of the most common ways to solve LASSO.

The idea is simple: update one coefficient at a time, holding all the others fixed. It turns out that each coordinate update has a closed-form solution.

Suppose we are updating coordinate $j$. We temporarily treat all other coefficients as fixed.

Define the partial residual

$$
r_j = y - \sum_{k \neq j} x_k w_k.
$$

This is the residual we would have if feature $j$ were removed from the current model and the coefficients were otherwise fixed.

With all other coefficients fixed, the only variable is $w_j$, so the LASSO problem reduces to the one-dimensional problem

$$
\min_{w_j} \frac{1}{2N}\|r_j - x_j w_j\|_2^2 + \lambda |w_j|.
$$

Here we use a denominator of $2N$ for convenience, but the main idea is unchanged by this scaling.

Let

$$
a_j = \frac{1}{N}x_j^\top x_j
$$

and

$$
c_j = \frac{1}{N}x_j^\top r_j.
$$

Then the coordinate descent update is

$$
w_j \leftarrow \frac{S(c_j,\lambda)}{a_j},
$$

where $S(c_j,\lambda)$ is the soft-thresholding operator.

To see where this update comes from, expand the one-dimensional objective:

$$
\frac{1}{2N}\|r_j - x_j w_j\|_2^2 + \lambda |w_j|.
$$

Expanding the squared error term gives

$$
\frac{1}{2N}(r_j - x_j w_j)^\top(r_j - x_j w_j) + \lambda |w_j|.
$$

So

$$
\frac{1}{2N}r_j^\top r_j - \frac{1}{N}w_j x_j^\top r_j + \frac{1}{2N}w_j^2 x_j^\top x_j + \lambda |w_j|.
$$

The first term does not depend on $w_j$, so we can ignore it when minimizing. Using the definitions of $a_j$ and $c_j$, the coordinate-wise objective is equivalent to minimizing

$$
g(w_j) = \frac{1}{2}a_j w_j^2 - c_j w_j + \lambda |w_j|.
$$

This is a one-dimensional convex problem. The only non-differentiable part is $|w_j|$, so we use the subgradient optimality condition.

The subgradient is

$$
\partial g(w_j) = a_j w_j - c_j + \lambda \partial |w_j|.
$$

A value of $w_j$ is optimal exactly when

$$
0 \in \partial g(w_j).
$$

Since

$$
\partial |w_j| =
\begin{cases}
\{1\}, & w_j > 0, \\
[-1,1], & w_j = 0, \\
\{-1\}, & w_j < 0,
\end{cases}
$$

we have

$$
\partial g(w_j) =
\begin{cases}
\{a_j w_j - c_j + \lambda\}, & w_j > 0, \\
[-c_j-\lambda,\,-c_j+\lambda], & w_j = 0, \\
\{a_j w_j - c_j - \lambda\}, & w_j < 0.
\end{cases}
$$

Now split according to the value of $c_j$.

### Case 1: $|c_j| \leq \lambda$

At $w_j = 0$,

$$
\partial g(0) = [-c_j-\lambda,\,-c_j+\lambda].
$$

The condition $|c_j| \leq \lambda$ is exactly the condition that this interval contains zero. Therefore,

$$
0 \in \partial g(0),
$$

so $w_j = 0$ is optimal.

Thus, if

$$
|c_j| \leq \lambda,
$$

then the coordinate update is

$$
w_j \leftarrow 0.
$$

### Case 2: $c_j > \lambda$

At $w_j = 0$,

$$
\partial g(0) = [-c_j-\lambda,\,-c_j+\lambda].
$$

Since $c_j > \lambda$,

$$
-c_j+\lambda < 0.
$$

So the entire subgradient interval at zero is negative, meaning zero is not optimal.

Also, no negative value can be optimal. If $w_j < 0$, then

$$
\partial g(w_j) = \{a_j w_j - c_j - \lambda\}.
$$

Because $a_jw_j < 0$, $c_j > 0$, and $\lambda > 0$, we have

$$
a_j w_j - c_j - \lambda < 0.
$$

So the subgradient cannot contain zero for any $w_j < 0$.

Therefore the minimizer must be positive. On the positive side,

$$
\partial g(w_j) = \{a_jw_j - c_j + \lambda\}.
$$

The optimality condition is

$$
a_jw_j - c_j + \lambda = 0.
$$

Solving gives

$$
w_j = \frac{c_j-\lambda}{a_j}.
$$

Thus, if

$$
c_j > \lambda,
$$

then the coordinate update is

$$
w_j \leftarrow \frac{c_j-\lambda}{a_j}.
$$

### Case 3: $c_j < -\lambda$

At $w_j = 0$,

$$
\partial g(0) = [-c_j-\lambda,\,-c_j+\lambda].
$$

Since $c_j < -\lambda$,

$$
-c_j-\lambda > 0.
$$

So the entire subgradient interval at zero is positive, meaning zero is not optimal.

Also, no positive value can be optimal. If $w_j > 0$, then

$$
\partial g(w_j) = \{a_jw_j - c_j + \lambda\}.
$$

Because $a_jw_j > 0$, $-c_j > \lambda$, and $\lambda > 0$, this quantity is positive. So the subgradient cannot contain zero for any $w_j > 0$.

Therefore the minimizer must be negative. On the negative side,

$$
\partial g(w_j) = \{a_jw_j - c_j - \lambda\}.
$$

The optimality condition is

$$
a_jw_j - c_j - \lambda = 0.
$$

Solving gives

$$
w_j = \frac{c_j+\lambda}{a_j}.
$$

Thus, if

$$
c_j < -\lambda,
$$

then the coordinate update is

$$
w_j \leftarrow \frac{c_j+\lambda}{a_j}.
$$



Combining the three cases,

$$
w_j
\leftarrow
\begin{cases}
\dfrac{c_j - \lambda}{a_j}, & c_j > \lambda, \\[1em]
0, & |c_j| \leq \lambda, \\[1em]
\dfrac{c_j + \lambda}{a_j}, & c_j < -\lambda.
\end{cases}
$$

This is exactly the soft-thresholding rule:

$$
w_j
\leftarrow
\frac{S(c_j,\lambda)}{a_j},
$$

The coordinate descent update comes directly from the subgradient optimality condition for the one-dimensional LASSO problem. If the coordinate-wise correlation $c_j$ is not large enough in magnitude to overcome the penalty $\lambda$, then zero satisfies the optimality condition.

To make this clear, assume that the columns of $X$ have been standardized so that

$$
\frac{1}{N}x_j^\top x_j = 1,
$$

then this simplifies to

$$
w_j
\leftarrow
S(c_j,\lambda).
$$

So, for standardized predictors, updating $w_j$ is just soft-thresholding the correlation between feature $j$ and the current partial residual.

The **coordinate descent algorithm** is:

1. Initialize $w$, often at $w=0$.
2. Cycle through coordinates $j=1,\dots,D$.
3. For each coordinate, compute the partial residual

   $$
   r_j = y - \sum_{k \neq j} x_k w_k.
   $$

4. Compute

   $$
   c_j = \frac{1}{N}x_j^\top r_j.
   $$
   and
   $$
   a_j = \frac{1}{N}x_j^\top x_j
    $$

6. Update

   $$
   w_j \leftarrow \frac{S(c_j,\lambda)}{a_j}.
   $$

7. Repeat until the coefficients stop changing.

The interpretation is: after accounting for the other variables, feature $j$ enters the model only if it is sufficiently correlated with the remaining residual.

If

$$
|c_j| \leq \lambda,
$$

then

$$
S(c_j,\lambda)=0,
$$

so the updated coefficient is exactly zero.

If

$$
|c_j| > \lambda,
$$

then the coefficient is nonzero, but shrunk toward zero.

Thus, coordinate descent makes the variable selection behavior of LASSO very explicit.

Let's look at a minimal implementation:

In [ ]:
import numpy as np
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler

In [ ]:
rng = np.random.default_rng()

N = 100
D = 20

X = rng.normal(size=(N, D))

w_true = np.zeros(D)
w_true[[1, 4, 7]] = [2.5, -1.8, 1.2]

y = X @ w_true + rng.normal(scale=0.7, size=N)

# Center y and standardize X.
# This lets us fit without an intercept.
X = StandardScaler().fit_transform(X)
y = y - y.mean()

Now we do the coordinate descent:

In [ ]:
def soft_threshold(c, lam):
    if c > lam:
        return c - lam
    elif c < -lam:
        return c + lam
    else:
        return 0.0

def lasso_coordinate_descent(X, y, lam, max_iter=1000, tol=1e-8):

    N, D = X.shape
    w = np.zeros(D)

    # a_j = (1/N) x_j^T x_j
    a = np.sum(X ** 2, axis=0) / N

    for iteration in range(max_iter):
        max_change = 0.0

        for j in range(D):
            old_w_j = w[j]

            # Current fitted values using all coordinates
            y_hat = X @ w

            # Partial residual:
            # r_j = y - sum_{k != j} x_k w_k
            #
            # Since y_hat = sum_k x_k w_k,
            # we remove all fitted values, then add back x_j w_j.
            r_j = y - y_hat + X[:, j] * old_w_j

            # c_j = (1/N) x_j^T r_j
            c_j = (X[:, j] @ r_j) / N

            # Coordinate update
            new_w_j = soft_threshold(c_j, lam) / a[j]

            # Store update
            w[j] = new_w_j

            max_change = max(max_change, abs(new_w_j - old_w_j))

        if max_change < tol:
            break

    return w, iteration + 1

In [ ]:
lam = 0.10

w_cd, n_iter = lasso_coordinate_descent(X, y, lam)

print(f"Coordinate descent finished in {n_iter} iterations")
print(np.round(w_cd, 3))

We can compare this to `sklearn`:

In [ ]:
lasso = Lasso(
    alpha=lam,
    fit_intercept=False,
    max_iter=10000,
    tol=1e-10
)

lasso.fit(X, y)

w_sklearn = lasso.coef_

print(np.round(w_sklearn, 3))

**Proximal gradient descent for LASSO**

Another way to solve LASSO is **proximal gradient descent**.

This method separates the objective into two parts:

$$
J(w)=
f(w) + g(w),
$$

where

$$
f(w)=
\frac{1}{2N}\|y - Xw\|_2^2
$$

is the smooth squared error part, and

$$
g(w)=
\lambda \|w\|_1
$$

is the non-smooth penalty part.

The gradient of the smooth part is

$$
\nabla f(w)=
-\frac{1}{N}X^\top(y - Xw).
$$

If we were only minimizing the squared error part, we could take an ordinary gradient descent step:

$$
u=
w - \eta \nabla f(w),
$$

where $\eta > 0$ is a step size.

For LASSO, we do not stop after this gradient step. Instead, we apply soft-thresholding:

$$
w^{new}=
S(u,\eta \lambda),
$$

where $S$ is applied coordinate by coordinate. This is called the **proximal** step. Essentially it is **projecting** the update onto the correct sparsity-inducing space. 

So proximal gradient descent for LASSO is:

1. Start with an initial vector $w$.
2. Take a gradient step on the squared error part:

   $$
   u=
   w - \eta \nabla f(w).
   $$

3. Apply soft-thresholding to every coordinate:

   $$
   w_j^{new}=
   S(u_j,\eta \lambda),
   \qquad j=1,\dots,D.
   $$

4. Repeat until convergence.

The interpretation is:

- the gradient step improves fit to the data
- the soft-thresholding step enforces sparsity

So proximal gradient descent alternates between reducing prediction error and pulling coefficients toward zero.

In [ ]:
import numpy as np

def soft_threshold_vector(u, lam):
    return np.sign(u) * np.maximum(np.abs(u) - lam, 0.0)

def lasso_proximal_gradient_descent(X, y, lam, eta=None, max_iter=1000, tol=1e-8):

    N, D = X.shape
    w = np.zeros(D)

    for iteration in range(max_iter):
        old_w = w.copy()

        # Current fitted values
        y_hat = X @ w

        # Current residual
        residual = y - y_hat

        # Gradient of the smooth squared-error part:
        #
        # f(w) = (1 / (2N)) ||y - Xw||_2^2
        #
        # grad f(w) = -(1 / N) X^T (y - Xw)
        grad = -(X.T @ residual) / N

        # Ordinary gradient descent step on the smooth part
        u = w - eta * grad

        # Proximal step for the L1 penalty
        w = soft_threshold_vector(u, eta * lam)

        # Check convergence
        max_change = np.max(np.abs(w - old_w))

        if max_change < tol:
            break

    return w, iteration + 1, eta

In [ ]:
from sklearn.linear_model import Lasso

lam = 0.10

w_pgd, n_iter, eta = lasso_proximal_gradient_descent(
    X, y, lam, max_iter=10000, tol=1e-8, eta = 1E-3
)

lasso = Lasso(
    alpha=lam,
    fit_intercept=False,
    max_iter=10000,
    tol=1e-10
)

lasso.fit(X, y)

w_sklearn = lasso.coef_

print(f"Proximal gradient descent finished in {n_iter} iterations")
print(f"Step size eta = {eta:.4f}")

print()
print("Proximal gradient coefficients:")
print(np.round(w_pgd, 3))

print()
print("sklearn coefficients:")
print(np.round(w_sklearn, 3))


## Constrained vs penalized

As with ridge, LASSO can be written in **penalized form**:

$$
\hat{w}=
\arg\min_w
\hat{R}(w)
+
\lambda \|w\|_1.
$$

There is an equivalent **constrained form**:

$$
\min_w \hat{R}(w)
\quad \text{s.t.} \quad
\|w\|_1 \le t.
$$

In the penalized form, $\lambda$ controls how much we penalize the total absolute size of the coefficients.

In the constrained form, $t$ sets a hard budget on the total absolute coefficient size.

These are two ways of parameterizing the same tradeoff:

- small $t$ corresponds to strong regularization
- large $t$ corresponds to weak regularization
- large $\lambda$ corresponds to strong regularization
- small $\lambda$ corresponds to weak regularization

The exact mapping between $\lambda$ and $t$ is usually not written down explicitly. But for each solution in one form, there is a corresponding value in the other form.

Another way to motivate LASSO is as a tractable approximation to an $L_0$-constrained problem. Ideally, we might want to solve

$$
\min_w \hat{R}(w)
\quad \text{s.t.} \quad
\|w\|_0 \le k,
$$

where $\|w\|_0$ counts the number of nonzero coefficients. This directly limits the model to use at most $k$ predictors. However, the $L_0$ constraint is nonconvex and generally difficult to optimize. LASSO replaces this hard sparsity constraint with the convex constraint $\|w\|_1 \le t$. The $L_1$ constraint does not count nonzero coefficients directly, but it encourages sparsity and is much easier to optimize.

### Geometry

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

try:
    from ipywidgets import interact, FloatSlider, Dropdown
except ImportError:
    interact = None
    FloatSlider = None
    Dropdown = None
from matplotlib.patches import Polygon
from sklearn.linear_model import Lasso

In [ ]:
X = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
    [2.0, 1.0]
])

y = np.array([1.0, 1.0, 2.0, 2.5])

w_ols = np.linalg.lstsq(X, y, rcond=None)[0]
w_ols

In [ ]:
def risk(w1, w2):
    # vectorized
    pred = X[:, 0, None, None] * w1 + X[:, 1, None, None] * w2
    residuals = y[:, None, None] - pred
    return np.mean(residuals**2, axis=0)

def penalized_risk(w1, w2, lam):
    # vectorized
    return risk(w1, w2) + lam * (np.abs(w1) + np.abs(w2))

def lasso_solution(lam):
    if lam == 0:
        return w_ols

    model = Lasso(
        alpha=lam,
        fit_intercept=False,
        max_iter=100000,
        tol=1e-10
    )
    model.fit(X, y)
    return model.coef_

def constrained_solution(t):
    # approximate correspondence between t and lambda

    ols_l1 = np.sum(np.abs(w_ols))
    if ols_l1 <= t:
        return w_ols, 0.0

    def l1_at_lambda(lam):
        w = lasso_solution(lam)
        return np.sum(np.abs(w))

    lo, hi = 0.0, 1.0
    while l1_at_lambda(hi) > t and hi < 1e6:
        hi *= 2.0

    # approximate binary search for lambda-star satisfying the constraint
    for _ in range(80):
        mid = 0.5 * (lo + hi)
        if l1_at_lambda(mid) > t:
            lo = mid
        else:
            hi = mid

    lam_star = 0.5 * (lo + hi)
    w_star = lasso_solution(lam_star)
    return w_star, lam_star

def l1_diamond(radius):
    return np.array([
        [ radius, 0.0],
        [0.0,  radius],
        [-radius, 0.0],
        [0.0, -radius]
    ])

def plot_geometry(mode="constrained", t=1.0, lam=0.2):
    xlim = (-2.5, 2.5)
    ylim = (-2.5, 2.5)

    fig, ax = plt.subplots(figsize=(7, 7))

    w1 = np.linspace(xlim[0], xlim[1], 400)
    w2 = np.linspace(ylim[0], ylim[1], 400)
    W1, W2 = np.meshgrid(w1, w2)

    R = risk(W1, W2)
    levels = np.linspace(R.min(), np.percentile(R, 90), 18)
    ax.contour(W1, W2, R, levels=levels, linewidths=1)

    ax.plot(w_ols[0], w_ols[1], "o", label="OLS")

    if mode == "constrained":
        w_star, lam_star = constrained_solution(t)

        diamond = Polygon(
            l1_diamond(t),
            closed=True,
            fill=False,
            linewidth=2,
            label=r"$\|w\|_1 \leq t$"
        )
        ax.add_patch(diamond)

        title = rf"LASSO constrained form: $t={t:.2f}$, approx $\lambda={lam_star:.3g}$"

    else:
        w_star = lasso_solution(lam)
        l1_radius = np.sum(np.abs(w_star))

        P = penalized_risk(W1, W2, lam)
        p_levels = np.linspace(P.min(), np.percentile(P, 85), 12)
        ax.contour(W1, W2, P, levels=p_levels, linestyles="dashed", linewidths=1)

        if l1_radius > 0:
            diamond = Polygon(
                l1_diamond(l1_radius),
                closed=True,
                fill=False,
                linewidth=2,
                label=rf"$\|w\|_1 = {l1_radius:.2f}$"
            )
            ax.add_patch(diamond)

        title = rf"LASSO penalized form: $\lambda={lam:.2f}$"

    ax.plot(w_star[0], w_star[1], "o", label="LASSO solution")

    ax.axhline(0, linewidth=0.8)
    ax.axvline(0, linewidth=0.8)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_xlabel(r"$w_1$")
    ax.set_ylabel(r"$w_2$")
    ax.set_title(title)
    ax.set_aspect("equal", adjustable="box")
    ax.legend()
    plt.show()

In [ ]:
if interact is not None:
    interact(
        plot_geometry,
        mode=Dropdown(
            options=["constrained", "penalized"],
            value="constrained",
            description="view"
        ),
        t=FloatSlider(
            value=1.0, min=0.05, max=4.0, step=0.05,
            description="t"
        ),
        lam=FloatSlider(
            value=0.2, min=0.0, max=5.0, step=0.05,
            description="lambda"
        )
    );
else:
    print("ipywidgets is not installed, so showing a static constrained-form plot.")
    plot_geometry(mode="constrained", t=1.0, lam=0.2)

Notice that OLS has elliptical loss contours. The unconstrained solution is the center of the smallest ellipse.

LASSO restricts us to lie inside an $\ell_1$ ball:

$$
\|w\|_1 \le t.
$$

In two dimensions, this is a diamond. The LASSO solution is the point where the smallest loss contour first touches the diamond.

This is similar to ridge, except the constraint region has corners. These corners lie on the coordinate axes. When the optimum occurs at one of these corners, one coefficient is exactly zero.

This geometric picture is the main reason LASSO produces sparse solutions.

- Ridge uses a round $\ell_2$ ball
- LASSO uses a diamond-shaped $\ell_1$ ball

Because the ridge ball is smooth, the point of tangency usually does not occur exactly on an axis. So ridge shrinks coefficients, but usually does not set them exactly to zero.

Because the LASSO ball has corners on the axes, the point of tangency often lands on an axis. This gives coefficients exactly equal to zero.

## LASSO and correlated variables

LASSO behaves differently from ridge when variables are highly correlated.

Suppose two features are nearly identical:

$$
x_2 \approx x_1.
$$

The model is

$$
y \approx w_1 x_1 + w_2 x_2.
$$

Since the two features contain almost the same information, many pairs $(w_1,w_2)$ lead to similar predictions. For ridge, the $\ell_2$ penalty often spreads weight across correlated variables.

LASSO often behaves differently. Because it prefers sparse solutions, it may choose one variable and set the other to zero:

$$
\hat{w}_1 \ne 0, \quad \hat{w}_2 = 0
$$

or

$$
\hat{w}_1 = 0, \quad \hat{w}_2 \ne 0.
$$

This is useful for variable selection, but it also means LASSO can be unstable when predictors are strongly correlated. Small changes in the data can change which variable is selected.

This is one motivation for the **elastic net**, which combines the $\ell_1$ and $\ell_2$ penalties. (We'll see this later.)

## Bias-variance tradeoff

LASSO provides another way to control the bias-variance tradeoff through $\lambda$.

- Small $\lambda$:
  - weak regularization
  - low bias
  - high variance
  - more selected variables

- Large $\lambda$:
  - strong regularization
  - high bias
  - low variance
  - fewer selected variables

As before, this leads to the familiar **U-shaped test error curve**. We can choose $\lambda$ using cross-validation.

Compared to ridge, LASSO also changes model complexity by changing the number of nonzero coefficients.

First consider a simulation with high variance and many useless variables:

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error

In [ ]:
np.random.seed(1)

N = 100
D = 50

X = np.random.normal(size=(N, D))

# True signal uses only a few features
w_true = np.zeros(D)
w_true[:5] = [3, -2, 1.5, 0.5, -1]

# relatively high noise
noise = np.random.normal(scale=2.0, size=N)
y = X @ w_true + noise

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, random_state=1
)

In [ ]:
lambdas = np.logspace(-4, 1, 250)

train_mse = []
test_mse = []
nonzero_counts = []
coef_l1_norms = []

for lam in lambdas:
    model = Lasso(
        alpha=lam,
        fit_intercept=False,
        max_iter=100000
    )
    model.fit(X_train, y_train)

    yhat_train = model.predict(X_train)
    yhat_test = model.predict(X_test)

    train_mse.append(mean_squared_error(y_train, yhat_train))
    test_mse.append(mean_squared_error(y_test, yhat_test))
    nonzero_counts.append(np.sum(model.coef_ != 0))
    coef_l1_norms.append(np.sum(np.abs(model.coef_)))

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(1/lambdas, train_mse, label="Training MSE")
plt.plot(1/lambdas, test_mse, label="Test MSE")
plt.xscale("log")
plt.xlabel(r"$1/\lambda$")
plt.ylabel("Mean squared error")
plt.title("Training and test error for LASSO regression")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(lambdas, nonzero_counts)
plt.xscale("log")
plt.xlabel(r"$\lambda$")
plt.ylabel("Number of nonzero coefficients")
plt.title("LASSO selects fewer variables as lambda increases")
plt.show()

As $\lambda$ increases, the number of selected variables decreases. This is different from ridge, where coefficients shrink toward zero but usually remain nonzero.

Now consider a simulation with high collinearity:

In [ ]:
np.random.seed(2)

N = 100
groups = 10
copies_per_group = 5
D = groups * copies_per_group

Z = np.random.normal(size=(N, groups))

X = np.hstack([
    Z[:, [j]] + 0.05 * np.random.normal(size=(N, copies_per_group))
    for j in range(groups)
])

w_true = np.zeros(D)
for j in range(groups):
    w_true[j * copies_per_group] = 1.0

y = X @ w_true + np.random.normal(scale=3.0, size=N)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, random_state=1
)

lambdas = np.logspace(-4, 1, 250)

train_mse = []
test_mse = []
nonzero_counts = []
coef_l1_norms = []

for lam in lambdas:
    model = Lasso(
        alpha=lam,
        fit_intercept=False,
        max_iter=100000
    )
    model.fit(X_train, y_train)

    train_mse.append(mean_squared_error(y_train, model.predict(X_train)))
    test_mse.append(mean_squared_error(y_test, model.predict(X_test)))
    nonzero_counts.append(np.sum(model.coef_ != 0))
    coef_l1_norms.append(np.sum(np.abs(model.coef_)))

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(1/lambdas, train_mse, label="Training MSE")
plt.plot(1/lambdas, test_mse, label="Test MSE")
plt.xscale("log")
plt.xlabel(r"$1/\lambda$")
plt.ylabel("Mean squared error")
plt.title("LASSO regression with highly collinear features")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(lambdas, nonzero_counts)
plt.xscale("log")
plt.xlabel(r"$\lambda$")
plt.ylabel("Number of nonzero coefficients")
plt.title("Selected variables under collinearity")
plt.show()

In the collinear setting, LASSO can improve test error by reducing variance, but the selected variables should be interpreted carefully.

If several variables carry almost the same information, LASSO may select one of them and ignore the others. The prediction can still be good, but the selected set is not necessarily unique or stable.

## Practical considerations

When applying LASSO in practice, there are a few important implementation details.

**Standardize predictors**

LASSO penalizes the size of coefficients:

$$
\lambda \|w\|_1 = \lambda \sum_{j=1}^D |w_j|.
$$

If predictors are on different scales, this penalty is not applied fairly.

For example:

- a variable with large scale may need a small coefficient
- a variable with small scale may need a large coefficient

Without standardization, LASSO may penalize some variables more heavily just because of their units.

Typically, we:

- center each feature
- scale each feature to unit variance

This ensures the penalty treats all predictors comparably.

We also typically **do not penalize the intercept**.

The intercept represents the baseline level of the response, not model complexity. Penalizing it can shrink the overall prediction level in an artificial way.

With an explicit intercept, the LASSO problem is

$$
\min_{\alpha,w}
\frac{1}{N}\|y - \alpha \mathbf{1} - Xw\|_2^2
+
\lambda \|w\|_1,
$$

where $\alpha$ is not penalized.

In practice, software usually handles this automatically if `fit_intercept=True`.

We choose $\lambda$ using validation or cross-validation. If we want an unbiased estimate of generalization error after tuning $\lambda$, we should use nested cross-validation.

Other practical points:

- LASSO is best suited to settings where sparse structure is plausible.
- It can be unstable with highly correlated predictors.
- It should not be treated as automatic causal variable selection.

## LASSO with orthonormal predictors

LASSO does not usually have a closed-form solution. But there is one useful special case.

Suppose the columns of $X$ are orthonormal after scaling, so that

$$
\frac{1}{N}X^\top X = I.
$$

Define

$$
z =
\frac{1}{N}X^\top y.
$$

In this case, $z_j$ is the OLS coefficient for feature $j$.

The LASSO solution becomes coordinate-wise:

$$
\hat{w}^{\lambda}_{lasso,j}=
S(z_j,\lambda),
$$

This shows the main effect of LASSO very clearly:

- if the OLS coefficient is small, LASSO sets it to zero
- if the OLS coefficient is large, LASSO shrinks it toward zero by $\lambda$

So LASSO is not just shrinkage. It is **thresholded shrinkage**.

We can visualize soft-thresholding directly:

In [ ]:
def soft_threshold(z, lam):
    return np.sign(z) * np.maximum(np.abs(z) - lam, 0.0)

z = np.linspace(-4, 4, 400)

lam = 1.0

plt.figure(figsize=(7, 5))
plt.plot(z, z, label="OLS")
plt.plot(z, soft_threshold(z, lam), label=rf"LASSO, $\lambda={lam:g}$")
plt.plot(z, z / (1 + lam), label=rf"Ridge, shrinkage $1/(1+\lambda)$, $\lambda={lam:g}$")

plt.axhline(0, linewidth=0.8)
plt.axvline(0, linewidth=0.8)
plt.xlabel(r"$z_j$")
plt.ylabel(r"$\hat{w}_{\lambda,j}$")
plt.title("LASSO soft-thresholding vs. ridge shrinkage")
plt.legend()
plt.show()

# Real Data Example

For a self-contained example, use the diabetes regression data from `sklearn`. The goal is to predict a quantitative disease progression measure from patient covariates.

This is not as high-dimensional as the gene-expression example, but it is useful for illustrating tuning, coefficient paths, and variable selection.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import mean_squared_error

In [ ]:
diabetes = load_diabetes()

X = diabetes.data
y = diabetes.target
feature_names = np.array(diabetes.feature_names)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("feature names:", feature_names)

In [ ]:
lambdas = np.logspace(-4, 2, 100)

outer_cv = KFold(n_splits=5, shuffle=True, random_state=164654)
inner_cv = KFold(n_splits=5, shuffle=True, random_state=56882)

lasso_pipeline = make_pipeline(
    StandardScaler(),
    Lasso(fit_intercept=True, max_iter=200000)
)

param_grid = {
    "lasso__alpha": lambdas
}

In [ ]:
outer_test_mse = []
outer_best_lambdas = []
outer_nonzero_counts = []
outer_l1_norms = []

for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X), start=1):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Inner CV chooses lambda using only the outer training fold
    search = GridSearchCV(
        estimator=lasso_pipeline,
        param_grid=param_grid,
        scoring="neg_mean_squared_error",
        cv=inner_cv,
        refit=True
    )

    search.fit(X_train, y_train)

    best_model = search.best_estimator_
    best_lambda = search.best_params_["lasso__alpha"]

    # Evaluate once on the outer test fold
    y_pred = best_model.predict(X_test)
    test_mse = mean_squared_error(y_test, y_pred)

    coef = best_model.named_steps["lasso"].coef_

    outer_test_mse.append(test_mse)
    outer_best_lambdas.append(best_lambda)
    outer_nonzero_counts.append(np.sum(coef != 0))
    outer_l1_norms.append(np.sum(np.abs(coef)))

    print(f"Fold {fold}")
    print(f"  best lambda: {best_lambda:.4g}")
    print(f"  outer test MSE: {test_mse:.4f}")
    print(f"  number nonzero: {np.sum(coef != 0)}")
    print(f"  L1 norm: {np.sum(np.abs(coef)):.4f}")

In [ ]:
outer_test_mse = np.array(outer_test_mse)
outer_best_lambdas = np.array(outer_best_lambdas)
outer_nonzero_counts = np.array(outer_nonzero_counts)
outer_l1_norms = np.array(outer_l1_norms)

print("Nested CV results")
print("-----------------")
print(f"Mean outer test MSE: {outer_test_mse.mean():.4f}")
print(f"Std outer test MSE:  {outer_test_mse.std(ddof=1):.4f}")
print(f"Mean nonzero count:  {outer_nonzero_counts.mean():.2f}")
print(f"Mean L1 norm:        {outer_l1_norms.mean():.4f}")

After estimating generalization error with nested CV, we can now tune $\lambda$ on all data and fit a final model.

In [ ]:
final_search = GridSearchCV(
    estimator=lasso_pipeline,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=inner_cv,
    refit=True
)

final_search.fit(X, y)

best_lambda_final = final_search.best_params_["lasso__alpha"]
final_model = final_search.best_estimator_

print("Final selected lambda:", best_lambda_final)

For illustration, we can now plot the CV curve from the final tuning step.

In [ ]:
cv_results = pd.DataFrame(final_search.cv_results_)

mean_cv_mse = -cv_results["mean_test_score"].to_numpy()
std_cv_mse = cv_results["std_test_score"].to_numpy()

plt.figure(figsize=(8, 5))
plt.plot(1/lambdas, mean_cv_mse, label="Mean CV MSE")
plt.fill_between(
    1/lambdas,
    mean_cv_mse - std_cv_mse,
    mean_cv_mse + std_cv_mse,
    alpha=0.2,
    label="+/- 1 std"
)

plt.axvline(
    1/best_lambda_final,
    linestyle="--",
    label=rf"Selected $\lambda$ = {best_lambda_final:.3g}"
)

plt.xscale("log")
plt.xlabel(r"$1/\lambda$")
plt.ylabel("Cross-validated MSE")
plt.title("LASSO tuning curve on full data")
plt.legend()
plt.show()

In [ ]:
final_lasso = final_model.named_steps["lasso"]
final_coef = final_lasso.coef_

coef_table = (
    pd.DataFrame({
        "feature": feature_names,
        "coef": final_coef,
        "abs_coef": np.abs(final_coef),
        "selected": final_coef != 0
    })
    .sort_values("abs_coef", ascending=False)
    .reset_index(drop=True)
)

print("Final number selected:", np.sum(final_coef != 0))
print("Final L1 norm:", np.sum(np.abs(final_coef)))
coef_table

Another useful pedagogical tool is the coefficient shrinkage curve. Let's refit over values of $\lambda$:

In [ ]:
coefs = []

for lam in lambdas:
    model = make_pipeline(
        StandardScaler(),
        Lasso(alpha=lam, fit_intercept=True, max_iter=200000)
    )

    model.fit(X, y)

    lasso_step = model.named_steps["lasso"]
    coefs.append(lasso_step.coef_)

coefs = np.array(coefs)
coef_l1_norms = np.sum(np.abs(coefs), axis=1)
nonzero_counts = np.sum(coefs != 0, axis=1)

then plot:

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(lambdas, coef_l1_norms)

#plt.xscale("log")
plt.xlabel(r"$\lambda$")
plt.ylabel(r"$\|\hat{w}_\lambda\|_1$")
plt.title("LASSO coefficient shrinkage")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(lambdas, nonzero_counts)

plt.xscale("log")
plt.xlabel(r"$log(\lambda)$")
plt.ylabel("Number of nonzero coefficients")
plt.title("LASSO sparsity path")
plt.show()

We can also plot individual coefficient paths:

In [ ]:
plt.figure(figsize=(9, 6))

for j in range(coefs.shape[1]):
    plt.plot(lambdas, coefs[:, j], label=feature_names[j])

plt.axhline(0, linewidth=0.8)
plt.xscale("log")
plt.xlabel(r"$log(\lambda)$")
plt.ylabel(r"$\hat{w}_{\lambda,j}$")
plt.title("LASSO coefficient paths")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

Notice that if we zoom in enough on any of these curves they are piecewise-linear on a linear-linear scale. This is a general feature of the LASSO path, its piecewise linear. 

In [ ]:
plt.figure(figsize=(9, 6))

k=2
n_lam_sel = 50
for j in range(k,k+1):
    plt.plot(lambdas[:n_lam_sel], coefs[:n_lam_sel, j], label=feature_names[j])
    plt.scatter(lambdas[:n_lam_sel], coefs[:n_lam_sel, j], label=feature_names[j])

#plt.axhline(0, linewidth=0.8)
#plt.xscale("log")
plt.xlabel(r"$\lambda$")
plt.ylabel(r"$\hat{w}_{\lambda,j}$")
plt.title("LASSO coefficient paths")
#plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

We can also compare to the OLS baseline. For a direct comparison, use the same outer cross-validation splits.

In [ ]:
ols_outer_test_mse = []
ols_outer_train_mse = []
ols_coef_l1_norms = []
ols_nonzero_counts = []

for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X), start=1):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    ols_model = make_pipeline(
        StandardScaler(),
        LinearRegression(fit_intercept=True)
    )

    ols_model.fit(X_train, y_train)

    yhat_train = ols_model.predict(X_train)
    yhat_test = ols_model.predict(X_test)

    ols_train_mse = mean_squared_error(y_train, yhat_train)
    ols_test_mse = mean_squared_error(y_test, yhat_test)

    ols_coef = ols_model.named_steps["linearregression"].coef_

    ols_outer_train_mse.append(ols_train_mse)
    ols_outer_test_mse.append(ols_test_mse)
    ols_coef_l1_norms.append(np.sum(np.abs(ols_coef)))
    ols_nonzero_counts.append(np.sum(ols_coef != 0))

ols_outer_train_mse = np.array(ols_outer_train_mse)
ols_outer_test_mse = np.array(ols_outer_test_mse)
ols_coef_l1_norms = np.array(ols_coef_l1_norms)
ols_nonzero_counts = np.array(ols_nonzero_counts)

print("OLS results")
print("-----------")
print(f"Mean train MSE:       {ols_outer_train_mse.mean():.4f}")
print(f"Mean outer test MSE:  {ols_outer_test_mse.mean():.4f}")
print(f"Std outer test MSE:   {ols_outer_test_mse.std(ddof=1):.4f}")
print(f"Mean nonzero count:   {ols_nonzero_counts.mean():.2f}")
print(f"Mean L1 norm:         {ols_coef_l1_norms.mean():.4f}")

print()

print("Nested CV LASSO results")
print("-----------------------")
print(f"Mean outer test MSE:  {outer_test_mse.mean():.4f}")
print(f"Std outer test MSE:   {outer_test_mse.std(ddof=1):.4f}")
print(f"Mean nonzero count:   {outer_nonzero_counts.mean():.2f}")
print(f"Mean L1 norm:         {outer_l1_norms.mean():.4f}")